# House Prices — Leakage-Free Stacking Notebook

This notebook is designed for **Kaggle upload** and follows a **leakage-aware stacking / blending** workflow.

## What is fixed compared with the original notebook?
The original notebook's final blend validation reused globally fitted models during CV.  
This notebook fixes that by:

- fitting preprocessing inside folds
- generating base-model OOF predictions fold-by-fold
- using **nested stacking evaluation** for the final ensemble
- training the final submission model on full training data only after OOF generation

## Outputs
- base model OOF RMSE
- leakage-free nested ensemble OOF RMSE
- `submission.csv`


In [ ]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import RidgeCV, Ridge
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

warnings.filterwarnings("ignore")

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False
    XGBRegressor = None

try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False
    LGBMRegressor = None

print("xgboost available:", HAS_XGB)
print("lightgbm available:", HAS_LGBM)


In [ ]:
# ============================================================
# CONFIG
# ============================================================
RANDOM_STATE = 42

# Runtime knobs
OUTER_FOLDS = 5
INNER_FOLDS = 5

# Set False if you want a faster Kaggle run focused only on submission
RUN_NESTED_EVAL = True

# Data path
# Kaggle competition notebook:
# /kaggle/input/house-prices-advanced-regression-techniques/
KAGGLE_INPUT = Path("/kaggle/input/house-prices-advanced-regression-techniques")
LOCAL_INPUT = Path("./data")

if KAGGLE_INPUT.exists():
    DATA_DIR = KAGGLE_INPUT
else:
    DATA_DIR = LOCAL_INPUT

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

print("Using data dir:", DATA_DIR)
print("train exists:", TRAIN_PATH.exists())
print("test exists :", TEST_PATH.exists())


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(train_df.shape, test_df.shape)
display(train_df.head())


In [ ]:
# ============================================================
# PREPROCESSOR
# ============================================================
CODE_AS_CATEGORY = ["MSSubClass", "YrSold", "MoSold"]
DISCRETE_EXCLUDE_FOR_LOG = [
    "MSSubClass", "KitchenAbvGr", "BedroomAbvGr", "TotRmsAbvGrd",
    "HalfBath", "FullBath", "BsmtHalfBath", "BsmtFullBath",
    "GarageCars", "Fireplaces", "MiscVal"
]

FIXED_FILL_STRING = {
    "Functional": "Typ",
    "Electrical": "SBrkr",
    "KitchenQual": "TA",
    "PoolQC": "None",
}

MODE_FILL_COLS = ["Exterior1st", "Exterior2nd", "SaleType"]
GARAGE_NUM_COLS = ["GarageYrBlt", "GarageArea", "GarageCars"]
GARAGE_CAT_COLS = ["GarageType", "GarageFinish", "GarageQual", "GarageCond"]
BSMT_CAT_COLS = ["BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"]


class AmesPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, skew_threshold=0.8):
        self.skew_threshold = skew_threshold

    def fit(self, X, y=None):
        df = X.copy()
        if "Id" in df.columns:
            df = df.drop(columns=["Id"])

        for col in CODE_AS_CATEGORY:
            if col in df.columns:
                df[col] = df[col].astype(str)

        self.columns_in_ = df.columns.tolist()

        self.mode_map_ = {}
        for col in MODE_FILL_COLS:
            if col in df.columns:
                mode = df[col].mode(dropna=True)
                self.mode_map_[col] = mode.iloc[0] if len(mode) > 0 else "None"

        self.mszoning_group_mode_ = {}
        self.mszoning_global_mode_ = "None"
        if "MSZoning" in df.columns and "MSSubClass" in df.columns:
            tmp = df[["MSSubClass", "MSZoning"]].copy()
            global_mode = tmp["MSZoning"].mode(dropna=True)
            self.mszoning_global_mode_ = global_mode.iloc[0] if len(global_mode) > 0 else "None"
            for key, sub in tmp.groupby("MSSubClass"):
                mode = sub["MSZoning"].mode(dropna=True)
                self.mszoning_group_mode_[str(key)] = mode.iloc[0] if len(mode) > 0 else self.mszoning_global_mode_

        self.lotfrontage_by_neighborhood_ = {}
        self.lotfrontage_global_median_ = 0.0
        if "LotFrontage" in df.columns:
            self.lotfrontage_global_median_ = float(df["LotFrontage"].median())
        if "LotFrontage" in df.columns and "Neighborhood" in df.columns:
            med = df.groupby("Neighborhood")["LotFrontage"].median()
            self.lotfrontage_by_neighborhood_ = med.to_dict()

        numeric = df.select_dtypes(include=[np.number]).copy()
        for col in DISCRETE_EXCLUDE_FOR_LOG:
            if col in numeric.columns:
                numeric = numeric.drop(columns=[col])

        self.skewed_cols_ = []
        for col in numeric.columns:
            if col == "SalePrice":
                continue
            if numeric[col].dropna().empty:
                continue
            if numeric[col].min(skipna=True) < 0:
                continue
            skew = numeric[col].skew()
            if pd.notna(skew) and abs(float(skew)) > self.skew_threshold:
                self.skewed_cols_.append(col)

        transformed = self._basic_transform(df, fit_mode=True)
        transformed = self._apply_log_transform(transformed)

        dummies = pd.get_dummies(transformed, drop_first=True)
        self.output_columns_ = dummies.columns.tolist()
        self.numeric_fill_values_ = {}
        for col in dummies.columns:
            if pd.api.types.is_numeric_dtype(dummies[col]):
                self.numeric_fill_values_[col] = float(dummies[col].median()) if not dummies[col].dropna().empty else 0.0
        return self

    def _fill_lotfrontage(self, value, neighborhood):
        if pd.notna(value):
            return value
        if neighborhood in self.lotfrontage_by_neighborhood_ and pd.notna(self.lotfrontage_by_neighborhood_[neighborhood]):
            return self.lotfrontage_by_neighborhood_[neighborhood]
        return self.lotfrontage_global_median_

    def _basic_transform(self, df, fit_mode=False):
        out = df.copy()

        for col in CODE_AS_CATEGORY:
            if col in out.columns:
                out[col] = out[col].astype(str)

        for col, fill_value in FIXED_FILL_STRING.items():
            if col in out.columns:
                out[col] = out[col].fillna(fill_value)

        for col in MODE_FILL_COLS:
            if col in out.columns:
                out[col] = out[col].fillna(self.mode_map_.get(col, "None"))

        if "MSZoning" in out.columns and "MSSubClass" in out.columns:
            def fill_zone(row):
                if pd.notna(row["MSZoning"]):
                    return row["MSZoning"]
                key = str(row["MSSubClass"])
                return self.mszoning_group_mode_.get(key, self.mszoning_global_mode_)
            out["MSZoning"] = out.apply(fill_zone, axis=1)

        for col in GARAGE_NUM_COLS:
            if col in out.columns:
                out[col] = out[col].fillna(0)

        for col in GARAGE_CAT_COLS + BSMT_CAT_COLS:
            if col in out.columns:
                out[col] = out[col].fillna("None")

        if "LotFrontage" in out.columns and "Neighborhood" in out.columns:
            out["LotFrontage"] = out.apply(
                lambda row: self._fill_lotfrontage(row["LotFrontage"], row["Neighborhood"]), axis=1
            )
            out["LotFrontage"] = out["LotFrontage"].fillna(self.lotfrontage_global_median_)

        obj_cols = out.select_dtypes(include="object").columns.tolist()
        if obj_cols:
            out[obj_cols] = out[obj_cols].fillna("None")

        num_cols = out.select_dtypes(include=[np.number]).columns.tolist()
        for col in num_cols:
            if out[col].isna().any():
                fill_value = 0.0 if fit_mode else self.numeric_fill_values_.get(col, 0.0)
                out[col] = out[col].fillna(fill_value)

        return out

    def _apply_log_transform(self, df):
        out = df.copy()
        for col in self.skewed_cols_:
            if col in out.columns:
                out[col] = np.log1p(out[col].clip(lower=0))
        return out

    def transform(self, X):
        df = X.copy()
        if "Id" in df.columns:
            df = df.drop(columns=["Id"])

        for col in self.columns_in_:
            if col not in df.columns:
                df[col] = np.nan
        df = df[self.columns_in_].copy()

        df = self._basic_transform(df, fit_mode=False)
        df = self._apply_log_transform(df)

        dummies = pd.get_dummies(df, drop_first=True)
        dummies = dummies.reindex(columns=self.output_columns_, fill_value=0)
        return dummies.astype(float)


def make_target(df):
    return np.log1p(df["SalePrice"]).copy()


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


In [ ]:
# ============================================================
# MODELS
# ============================================================
def get_base_model_builders(random_state=42):
    builders = {
        "ridge": lambda: make_pipeline(
            RobustScaler(),
            RidgeCV(
                alphas=[
                    1e-15, 1e-10, 1e-8, 9e-4, 7e-4, 5e-4, 3e-4, 1e-4,
                    1e-3, 5e-2, 1e-2, 0.1, 0.3, 1, 3, 5, 10, 15, 18, 20, 30, 50, 75, 100
                ],
                cv=5,
            ),
        ),
        "svr": lambda: make_pipeline(
            RobustScaler(),
            SVR(C=20, epsilon=0.008, gamma=0.0003),
        ),
        "gbr": lambda: GradientBoostingRegressor(
            n_estimators=3000,
            learning_rate=0.01,
            max_depth=4,
            max_features="sqrt",
            min_samples_leaf=15,
            min_samples_split=10,
            loss="huber",
            random_state=random_state,
        ),
        "rf": lambda: RandomForestRegressor(
            n_estimators=500,
            max_depth=15,
            min_samples_split=5,
            min_samples_leaf=5,
            random_state=random_state,
            n_jobs=-1,
        ),
    }

    if HAS_XGB:
        builders["xgboost"] = lambda: XGBRegressor(
            tree_method="hist",
            learning_rate=0.02,
            n_estimators=1500,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.0,
            reg_lambda=1.0,
            objective="reg:squarederror",
            random_state=random_state,
            n_jobs=1,
        )

    if HAS_LGBM:
        builders["lightgbm"] = lambda: LGBMRegressor(
            objective="regression",
            learning_rate=0.02,
            num_leaves=16,
            n_estimators=1500,
            subsample=0.8,
            colsample_bytree=0.7,
            reg_alpha=0.0,
            reg_lambda=0.0,
            min_child_samples=20,
            random_state=random_state,
            verbose=-1,
        )
    return builders


def get_meta_model_builder(random_state=42):
    return lambda: Ridge(alpha=1.0, random_state=random_state)


BASE_BLEND_WEIGHTS = {
    "ridge": 0.10,
    "svr": 0.15,
    "gbr": 0.15,
    "rf": 0.10,
    "xgboost": 0.15,
    "lightgbm": 0.15,
}
STACK_WEIGHT = 0.20


def normalized_weights(active_names):
    raw = {name: BASE_BLEND_WEIGHTS.get(name, 0.0) for name in active_names}
    total = sum(raw.values())
    if total <= 0:
        raw = {name: 1.0 for name in active_names}
        total = sum(raw.values())
    base_target = 1.0 - STACK_WEIGHT
    return {name: (value / total) * base_target for name, value in raw.items()}


In [ ]:
# ============================================================
# OOF + ENSEMBLE HELPERS
# ============================================================
def generate_base_oof_predictions(raw_X, y, n_splits=5, random_state=42):
    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    builders = get_base_model_builders(random_state=random_state)

    oof = {name: np.zeros(len(raw_X), dtype=float) for name in builders}
    fold_scores = {name: [] for name in builders}

    for fold_no, (train_idx, valid_idx) in enumerate(cv.split(raw_X, y), start=1):
        X_tr_raw = raw_X.iloc[train_idx].reset_index(drop=True)
        X_va_raw = raw_X.iloc[valid_idx].reset_index(drop=True)
        y_tr = y.iloc[train_idx].reset_index(drop=True)
        y_va = y.iloc[valid_idx].reset_index(drop=True)

        preprocessor = AmesPreprocessor()
        X_tr = preprocessor.fit_transform(X_tr_raw)
        X_va = preprocessor.transform(X_va_raw)

        for name, builder in builders.items():
            model = builder()
            model.fit(X_tr, y_tr)
            pred = model.predict(X_va)
            oof[name][valid_idx] = pred
            fold_scores[name].append(rmse(y_va, pred))

    scores = {
        name: {
            "fold_rmse": vals,
            "mean_rmse": float(np.mean(vals)),
            "std_rmse": float(np.std(vals)),
        }
        for name, vals in fold_scores.items()
    }
    return pd.DataFrame(oof), scores


def fit_full_base_models(raw_X_train, y, raw_X_test, random_state=42):
    builders = get_base_model_builders(random_state=random_state)
    preprocessor = AmesPreprocessor()
    X_train = preprocessor.fit_transform(raw_X_train)
    X_test = preprocessor.transform(raw_X_test)

    fitted_models = {}
    test_preds = {}
    for name, builder in builders.items():
        model = builder()
        model.fit(X_train, y)
        fitted_models[name] = model
        test_preds[name] = model.predict(X_test)

    return preprocessor, fitted_models, pd.DataFrame(test_preds)


def evaluate_nested_stacking_ensemble(raw_X, y, outer_splits=5, inner_splits=5, random_state=42):
    outer_cv = KFold(n_splits=outer_splits, shuffle=True, random_state=random_state)
    builders = get_base_model_builders(random_state=random_state)
    active_names = list(builders.keys())
    base_weights = normalized_weights(active_names)
    meta_builder = get_meta_model_builder(random_state=random_state)

    oof_base = {name: np.zeros(len(raw_X), dtype=float) for name in active_names}
    oof_stack = np.zeros(len(raw_X), dtype=float)
    oof_blend = np.zeros(len(raw_X), dtype=float)
    fold_metrics = []

    for fold_no, (train_idx, valid_idx) in enumerate(outer_cv.split(raw_X, y), start=1):
        raw_outer_train = raw_X.iloc[train_idx].reset_index(drop=True)
        raw_outer_valid = raw_X.iloc[valid_idx].reset_index(drop=True)
        y_outer_train = y.iloc[train_idx].reset_index(drop=True)
        y_outer_valid = y.iloc[valid_idx].reset_index(drop=True)

        # Inner OOF features for meta-model training
        inner_meta_X, _ = generate_base_oof_predictions(
            raw_outer_train, y_outer_train,
            n_splits=inner_splits,
            random_state=random_state + fold_no
        )
        meta_model = meta_builder()
        meta_model.fit(inner_meta_X, y_outer_train)

        # Full outer-train fit for base models
        preprocessor = AmesPreprocessor()
        X_outer_train = preprocessor.fit_transform(raw_outer_train)
        X_outer_valid = preprocessor.transform(raw_outer_valid)

        valid_base = {}
        for name, builder in builders.items():
            model = builder()
            model.fit(X_outer_train, y_outer_train)
            pred = model.predict(X_outer_valid)
            oof_base[name][valid_idx] = pred
            valid_base[name] = pred

        valid_base = pd.DataFrame(valid_base)
        stack_pred = meta_model.predict(valid_base)

        blend_pred = np.zeros(len(valid_base), dtype=float)
        for name in active_names:
            blend_pred += base_weights[name] * valid_base[name].values
        blend_pred += STACK_WEIGHT * stack_pred

        oof_stack[valid_idx] = stack_pred
        oof_blend[valid_idx] = blend_pred

        fold_metrics.append({
            "fold": fold_no,
            "rmse_stack": rmse(y_outer_valid, stack_pred),
            "rmse_blend": rmse(y_outer_valid, blend_pred),
        })

    summary = {
        "base_weights_normalized": base_weights,
        "stack_weight": STACK_WEIGHT,
        "fold_metrics": fold_metrics,
        "overall_rmse_stack": rmse(y, oof_stack),
        "overall_rmse_blend": rmse(y, oof_blend),
    }
    return {
        "oof_base": pd.DataFrame(oof_base),
        "oof_stack": oof_stack,
        "oof_blend": oof_blend,
        "summary": summary,
    }


def fit_final_ensemble_and_predict(raw_X_train, y, raw_X_test, inner_splits=5, random_state=42):
    builders = get_base_model_builders(random_state=random_state)
    active_names = list(builders.keys())
    base_weights = normalized_weights(active_names)
    meta_builder = get_meta_model_builder(random_state=random_state)

    train_meta_X, base_scores = generate_base_oof_predictions(
        raw_X_train, y, n_splits=inner_splits, random_state=random_state
    )

    preprocessor, fitted_base_models, test_base_preds = fit_full_base_models(
        raw_X_train, y, raw_X_test, random_state=random_state
    )

    meta_model = meta_builder()
    meta_model.fit(train_meta_X, y)
    stack_test_pred = meta_model.predict(test_base_preds)

    blend_test_pred = np.zeros(len(test_base_preds), dtype=float)
    for name in active_names:
        blend_test_pred += base_weights[name] * test_base_preds[name].values
    blend_test_pred += STACK_WEIGHT * stack_test_pred

    return {
        "preprocessor": preprocessor,
        "base_models": fitted_base_models,
        "meta_model": meta_model,
        "base_scores": base_scores,
        "active_model_names": active_names,
        "base_weights": base_weights,
        "stack_weight": STACK_WEIGHT,
        "test_base_predictions": test_base_preds,
        "test_stack_prediction": stack_test_pred,
        "test_blend_prediction": blend_test_pred,
    }


In [ ]:
# ============================================================
# PREPARE DATA
# ============================================================
raw_X_train = train_df.drop(columns=["SalePrice"]).copy()
raw_X_test = test_df.copy()
y = make_target(train_df)

print("Raw train shape:", raw_X_train.shape)
print("Raw test shape :", raw_X_test.shape)
print("Target shape   :", y.shape)


In [ ]:
# ============================================================
# BASE MODEL OOF RESULTS
# ============================================================
base_oof, base_scores = generate_base_oof_predictions(
    raw_X_train, y, n_splits=INNER_FOLDS, random_state=RANDOM_STATE
)

base_result_table = pd.DataFrame({
    name: {
        "mean_rmse": info["mean_rmse"],
        "std_rmse": info["std_rmse"],
    }
    for name, info in base_scores.items()
}).T.sort_values("mean_rmse")

display(base_result_table)


In [ ]:
# ============================================================
# LEAKAGE-FREE NESTED ENSEMBLE EVALUATION
# ============================================================
nested_result = None

if RUN_NESTED_EVAL:
    nested_result = evaluate_nested_stacking_ensemble(
        raw_X_train,
        y,
        outer_splits=OUTER_FOLDS,
        inner_splits=INNER_FOLDS,
        random_state=RANDOM_STATE,
    )
    print(json.dumps(nested_result["summary"], indent=2))
else:
    print("Skipped nested evaluation for speed.")


In [ ]:
# ============================================================
# FINAL FIT + TEST PREDICTION
# ============================================================
final_result = fit_final_ensemble_and_predict(
    raw_X_train,
    y,
    raw_X_test,
    inner_splits=INNER_FOLDS,
    random_state=RANDOM_STATE,
)

submission = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": np.expm1(final_result["test_blend_prediction"]),
})

display(submission.head())
print("submission shape:", submission.shape)


In [ ]:
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")


In [ ]:
summary = {
    "base_scores": base_scores,
    "nested_summary": None if nested_result is None else nested_result["summary"],
    "active_model_names": final_result["active_model_names"],
    "base_weights": final_result["base_weights"],
    "stack_weight": final_result["stack_weight"],
}

with open("training_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Saved training_summary.json")
summary
